# 02 — Claude API Headline Sentiment (Moonshot)

**Owner**: Person C (NLP Moonshot)  
**Phase**: 2 (Hours 3–6)  
**HARD TIMEBOX**: 3 hours max. If no signal by Hour 6, STOP and redeploy to OHLC features.

**Goal**: Determine if Claude's semantic understanding extracts signal from seen headlines
that keyword matching cannot.

**Decision gate**: If correlation of Claude sentiment (seen-only!) with target return is
r < 0.05 or p > 0.10 → headlines are fundamentally dead. STOP.

**Cost budget**: ~$2 for training headlines, ~$10-15 if we process test headlines.

**CRITICAL**: Only use SEEN headlines (bar_ix 0-49). The unseen headlines (bar_ix 50-99)
have strong signal (r=0.244) but are unavailable at test time. Mixing them in will give
a misleadingly optimistic result.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from scipy import stats

## 1. Setup Anthropic Client

In [ ]:
# TODO: pip install anthropic (if not already installed)
# TODO: import anthropic
# TODO: client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
# TODO: Verify connectivity with a small test call

## 2. Load Seen Headlines Only

In [ ]:
# TODO: Load headlines_seen_train.parquet (~9,740 headlines)
# TODO: Load bars for target computation
# TODO: Compute target return per session
# TODO: Print headline count, session count

## 3. Batch Classify Headlines with Claude Haiku

Strategy: Send batches of 10 headlines per request.  
Model: `claude-haiku-4-5-20251001` for cost efficiency.  
Expected: ~1,000 requests, ~$2, ~20 min with parallel requests.

In [ ]:
# TODO: Define the classification prompt:
#   "Rate each headline's sentiment for the stock it mentions.
#    Use this scale: -2 (very negative), -1 (negative), 0 (neutral),
#    +1 (positive), +2 (very positive).
#    Return ONLY the numbers separated by commas, one per headline."
#
# TODO: Define async batch processing function:
#   - Group headlines into batches of 10
#   - Send each batch as a Claude API call
#   - Parse response into list of sentiment scores
#   - Use asyncio / concurrent.futures for parallelism
#   - Add retry logic for rate limits
#   - Track cost (input/output tokens)
#
# TODO: Run on all ~9,740 training headlines
# TODO: Save results to notebooks/claude_sentiment_train.csv
#        (columns: session, bar_ix, headline, claude_sentiment)

## 4. Decision Gate: Does Claude Sentiment Have Signal?

In [ ]:
# TODO: Load claude sentiment results
# TODO: Aggregate per session: mean, sum, std of claude_sentiment
# TODO: Compute correlation with target return (SEEN HEADLINES ONLY!)
# TODO: Print r and p-value
#
# DECISION GATE:
#   if r < 0.05 or p > 0.10:
#       print("STOP — Headlines are dead. Redeploy to OHLC features.")
#   else:
#       print(f"SIGNAL FOUND: r={r:.4f}, p={p:.4f} — proceed to test data.")
#
# TODO: Also compare to keyword baseline (r=0.018, p=0.571)
#        Does Claude beat keyword matching?

## 5. (If Signal Found) Process Test Headlines

Only run this section if the decision gate passed.

In [ ]:
# TODO: Load headlines_seen_public_test.parquet (~99K headlines)
# TODO: Load headlines_seen_private_test.parquet (~99K headlines)
# TODO: Run same batch classification on both
# TODO: Save to notebooks/claude_sentiment_public_test.csv
# TODO: Save to notebooks/claude_sentiment_private_test.csv
# TODO: Print cost summary (total tokens, total $)
# TODO: Print time elapsed

## 6. Integration with Feature Pipeline

If Claude sentiment has signal, add it as a feature in `src/features.py`.

In [ ]:
# TODO: Merge claude_sentiment into the feature matrix
# TODO: Re-run cross_validate with the new feature included
# TODO: Compare CV Sharpe: with vs without claude_sentiment
# TODO: If improvement > 0.05 Sharpe AND stable across 20 seeds:
#   TODO: Update src/features.py to load and merge claude sentiment CSVs
#   TODO: Regenerate submissions
#   TODO: Add to leaderboard

## 7. Cost & Time Summary

In [ ]:
# TODO: Print total API calls made
# TODO: Print total tokens (input + output)
# TODO: Print total cost in $
# TODO: Print total wall-clock time
# TODO: Print verdict: SIGNAL / NO SIGNAL